# pSCENIC regulon activity analysis for malignant epithelial subclusters

This notebook uses the curated malignant epithelial object and the existing pySCENIC output loom to quantify predicted TF regulon activity across malignant epithelial subclusters.

Inputs:
- `GSE131907_GSE253013_mEpi_anno.h5ad`: malignant epithelial subcluster annotation (`Cell_type2`).
- `GSE131907_GSE253013_mEpi_out.loom`: pySCENIC AUCell output.
- `mEpi_cells_reg.csv` and `mEpi_cells_adj.sample.tsv`: retained for provenance.

Outputs are written to `results/pyscenic_malignant/`.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import warnings

import anndata as ad
import loompy
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import ranksums

from pyscenic.rss import regulon_specificity_scores

warnings.filterwarnings('ignore')

BASE_DIR = Path('/data/work/2026317luad')
OUT_DIR = BASE_DIR / 'results' / 'pyscenic_malignant'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MALIGNANT_H5AD = BASE_DIR / 'GSE131907_GSE253013_mEpi_anno.h5ad'
PYSCENIC_LOOM = BASE_DIR / 'GSE131907_GSE253013_mEpi_out.loom'
CTX_REGULONS_CSV = BASE_DIR / 'mEpi_cells_reg.csv'
ADJACENCIES_TSV = BASE_DIR / 'mEpi_cells_adj.sample.tsv'

CLUSTER_COL = 'Cell_type2'
RANDOM_STATE = 2024
TOP_N = 15

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=150, frameon=False)
print('Output:', OUT_DIR)

Output: /data/work/2026317luad/results/pyscenic_malignant


/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom 

## Helper functions

In [2]:
def clean_regulon_name(name: str) -> str:
    return str(name).replace('(+)', '').replace('(-)', '')


def load_auc_from_pyscenic_loom(loom_path: Path) -> pd.DataFrame:
    """Load pySCENIC AUCell matrix from a loom file.

    pySCENIC writes structured loom attributes that are useful but may fail strict
    loom validation, so validate=False is intentional here.
    """
    with loompy.connect(str(loom_path), mode='r', validate=False) as ds:
        auc_struct = ds.ca['RegulonsAUC']
        regulon_names = list(auc_struct.dtype.names)
        if 'CellID' in ds.ca.keys():
            cell_ids = ds.ca['CellID'].astype(str)
        elif 'obs_names' in ds.ca.keys():
            cell_ids = ds.ca['obs_names'].astype(str)
        else:
            raise KeyError('No CellID/obs_names column attribute found in loom.')
        auc_mtx = pd.DataFrame(
            {clean_regulon_name(r): auc_struct[r].astype('float32') for r in regulon_names},
            index=pd.Index(cell_ids, name='cell_id'),
        )
    # If cleaning creates duplicate regulon names, keep the highest AUC per TF.
    auc_mtx = auc_mtx.T.groupby(level=0).max().T
    return auc_mtx


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    p = pvals[ok]
    order = np.argsort(p)
    ranked = p[order]
    n = len(ranked)
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    ok_idx = np.where(ok)[0]
    qvals[ok_idx[order]] = q
    return qvals

## Load malignant epithelial annotation and regulon AUC

In [3]:
adata = sc.read_h5ad(MALIGNANT_H5AD)
if CLUSTER_COL not in adata.obs.columns:
    raise KeyError(f'{CLUSTER_COL} not found in adata.obs')

auc_mtx = load_auc_from_pyscenic_loom(PYSCENIC_LOOM)
common = adata.obs_names.intersection(auc_mtx.index)
print('adata:', adata.shape)
print('AUC matrix before align:', auc_mtx.shape)
print('Common cells:', len(common))

adata = adata[common].copy()
auc_mtx = auc_mtx.loc[common].copy()

cluster_order = adata.obs[CLUSTER_COL].astype(str).value_counts().index.tolist()
adata.obs[CLUSTER_COL] = pd.Categorical(adata.obs[CLUSTER_COL].astype(str), categories=cluster_order, ordered=True)

cell_counts = adata.obs[CLUSTER_COL].value_counts().loc[cluster_order].rename('n_cells').reset_index()
cell_counts.columns = [CLUSTER_COL, 'n_cells']
cell_counts.to_csv(OUT_DIR / 'malignant_subcluster_cell_counts.csv', index=False)
cell_counts

adata: (21548, 3000)
AUC matrix before align: (21548, 239)
Common cells: 21548


,Cell_type2,n_cells
0,C15orf48+ Epi,5330
1,RPS15A+ Epi,4306
2,CAPS+ Epi,3815
3,SCGB3A1+ Epi,2600
4,RPL35A+ Epi,1398
5,MARCKSL1+ Epi,1189
6,RPL32+ Epi,748
7,RPS15+ Epi,740
8,TM4SF1+ Epi,689
9,RPL10+ Epi,585


## Save a compact regulon-AUC AnnData object

In [4]:
meta_cols = [c for c in [CLUSTER_COL, 'Sample', 'dataset', 'Group_simple', 'Group2', 'cnv_status', 'Cell_type', 'Cell_subtype'] if c in adata.obs.columns]
regulon_adata = ad.AnnData(
    X=auc_mtx.to_numpy(dtype='float32'),
    obs=adata.obs[meta_cols].copy(),
    var=pd.DataFrame(index=pd.Index(auc_mtx.columns, name='regulon')),
)
regulon_adata.uns['source_files'] = {
    'malignant_h5ad': str(MALIGNANT_H5AD),
    'pyscenic_loom': str(PYSCENIC_LOOM),
    'ctx_regulons_csv': str(CTX_REGULONS_CSV),
    'adjacencies_tsv': str(ADJACENCIES_TSV),
}
regulon_adata.write_h5ad(OUT_DIR / 'malignant_regulon_auc.h5ad', compression='gzip')
auc_mtx.to_csv(OUT_DIR / 'malignant_regulon_auc.csv.gz')
print(regulon_adata)

AnnData object with n_obs × n_vars = 21548 × 239
    obs: 'Cell_type2', 'Sample', 'dataset', 'Group_simple', 'Group2', 'cnv_status', 'Cell_type', 'Cell_subtype'
    uns: 'source_files'


## Cluster-level regulon activity summary

In [5]:
auc_with_cluster = auc_mtx.join(adata.obs[[CLUSTER_COL]])
cluster_mean = auc_with_cluster.groupby(CLUSTER_COL, observed=True).mean().loc[cluster_order]
cluster_median = auc_with_cluster.groupby(CLUSTER_COL, observed=True).median().loc[cluster_order]
cluster_z = cluster_mean.apply(lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-8), axis=0)

cluster_mean.to_csv(OUT_DIR / 'regulon_cluster_mean_auc.csv')
cluster_median.to_csv(OUT_DIR / 'regulon_cluster_median_auc.csv')
cluster_z.to_csv(OUT_DIR / 'regulon_cluster_zscore_auc.csv')

cluster_z.iloc[:5, :5]

,ALX1,ARID3A,ASCL1,ATF3,ATF4
Cell_type2,,,,,
C15orf48+ Epi,-0.638777,-0.317267,-0.576055,2.204463,1.763668
RPS15A+ Epi,-0.063521,-0.335598,-0.415422,-0.669239,-0.742679
CAPS+ Epi,0.090232,-0.275582,-0.777897,-1.335469,-2.438805
SCGB3A1+ Epi,-0.259236,-0.273809,2.746347,0.018095,0.041243
RPL35A+ Epi,3.089770,3.155798,-0.168624,-1.230298,-0.205391


## Regulon specificity score (RSS)

RSS summarizes subcluster specificity of predicted regulon activity. Higher values indicate stronger specificity to that malignant epithelial subcluster.

In [6]:
rss = regulon_specificity_scores(auc_mtx, adata.obs[CLUSTER_COL])
rss.to_csv(OUT_DIR / 'regulon_specificity_scores.csv')

rows = []
for cluster in cluster_order:
    if cluster in rss.index:
        series = rss.loc[cluster]
    elif cluster in rss.columns:
        series = rss[cluster]
    else:
        continue
    for rank, (regulon, score) in enumerate(series.sort_values(ascending=False).head(TOP_N).items(), start=1):
        rows.append({'cluster': cluster, 'rank': rank, 'regulon': regulon, 'RSS': float(score)})

top_rss = pd.DataFrame(rows)
top_rss.to_csv(OUT_DIR / 'top_regulons_by_rss.csv', index=False)
top_rss.head(30)

,cluster,rank,regulon,RSS
0,C15orf48+ Epi,1,TSC22D4,0.493476
1,C15orf48+ Epi,2,PITX1,0.477894
2,C15orf48+ Epi,3,FOSL1,0.465756
3,C15orf48+ Epi,4,HSF1,0.457201
4,C15orf48+ Epi,5,CEBPB,0.448749
5,C15orf48+ Epi,6,FOXD1,0.448255
6,C15orf48+ Epi,7,MAFG,0.447036
7,C15orf48+ Epi,8,PPARG,0.439717
8,C15orf48+ Epi,9,FOXA3,0.437960
9,C15orf48+ Epi,10,CEBPD,0.436213


## Differential regulon activity: each subcluster versus all other malignant cells

This is a statistical screen for predicted regulon activity differences, not a causal TF mechanism test.

In [7]:
labels = adata.obs[CLUSTER_COL].astype(str).to_numpy()
auc_arr = auc_mtx.to_numpy(dtype='float32')
rows = []

for cluster in cluster_order:
    in_mask = labels == cluster
    out_mask = ~in_mask
    for j, regulon in enumerate(auc_mtx.columns):
        x = auc_arr[in_mask, j]
        y = auc_arr[out_mask, j]
        stat, pval = ranksums(x, y)
        rows.append({
            'cluster': cluster,
            'regulon': regulon,
            'n_in': int(in_mask.sum()),
            'n_out': int(out_mask.sum()),
            'mean_in': float(np.mean(x)),
            'mean_out': float(np.mean(y)),
            'delta_mean_auc': float(np.mean(x) - np.mean(y)),
            'z_stat': float(stat),
            'pvalue': float(pval),
        })

diff = pd.DataFrame(rows)
diff['fdr_bh'] = bh_fdr(diff['pvalue'].to_numpy())
diff = diff.sort_values(['cluster', 'fdr_bh', 'delta_mean_auc'], ascending=[True, True, False])
diff.to_csv(OUT_DIR / 'regulon_differential_activity_ranksums.csv', index=False)

top_diff = diff[(diff['fdr_bh'] < 0.05) & (diff['delta_mean_auc'] > 0)].groupby('cluster').head(TOP_N)
top_diff.to_csv(OUT_DIR / 'top_regulons_by_differential_activity.csv', index=False)
top_diff.head(30)

,cluster,regulon,n_in,n_out,mean_in,mean_out,delta_mean_auc,z_stat,pvalue,fdr_bh
198,C15orf48+ Epi,TSC22D4,5330,16218,0.175919,0.053640,0.122279,67.525269,0.000000e+00,0.000000e+00
78,C15orf48+ Epi,HMGA1,5330,16218,0.319661,0.218890,0.100771,40.868313,0.000000e+00,0.000000e+00
13,C15orf48+ Epi,CEBPD,5330,16218,0.242949,0.149323,0.093626,53.902274,0.000000e+00,0.000000e+00
12,C15orf48+ Epi,CEBPB,5330,16218,0.151914,0.090100,0.061814,80.655832,0.000000e+00,0.000000e+00
156,C15orf48+ Epi,PITX1,5330,16218,0.117421,0.056115,0.061306,101.927112,0.000000e+00,0.000000e+00
112,C15orf48+ Epi,JUNB,5330,16218,0.139278,0.101805,0.037473,67.464734,0.000000e+00,0.000000e+00
99,C15orf48+ Epi,HSF1,5330,16218,0.074168,0.038836,0.035332,75.939872,0.000000e+00,0.000000e+00
5,C15orf48+ Epi,BATF,5330,16218,0.098959,0.065473,0.033486,62.001401,0.000000e+00,0.000000e+00
124,C15orf48+ Epi,MAFG,5330,16218,0.079967,0.046959,0.033008,87.820139,0.000000e+00,0.000000e+00
47,C15orf48+ Epi,FOSL1,5330,16218,0.060249,0.029015,0.031235,75.841112,0.000000e+00,0.000000e+00


## Save run information

In [8]:
info = {
    'python': sys.version,
    'scanpy': sc.__version__,
    'n_cells': int(adata.n_obs),
    'n_regulons': int(auc_mtx.shape[1]),
    'cluster_col': CLUSTER_COL,
    'cluster_order': cluster_order,
}
try:
    import pyscenic
    info['pyscenic'] = pyscenic.__version__
except Exception:
    pass

(OUT_DIR / 'run_info.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
print(json.dumps(info, indent=2))

{
  "python": "3.10.12 | packaged by conda-forge | (main, Jun 23 2023, 22:40:32) [GCC 12.3.0]",
  "scanpy": "1.9.5",
  "n_cells": 21548,
  "n_regulons": 239,
  "cluster_col": "Cell_type2",
  "cluster_order": [
    "C15orf48+ Epi",
    "RPS15A+ Epi",
    "CAPS+ Epi",
    "SCGB3A1+ Epi",
    "RPL35A+ Epi",
    "MARCKSL1+ Epi",
    "RPL32+ Epi",
    "RPS15+ Epi",
    "TM4SF1+ Epi",
    "RPL10+ Epi",
    "TAGLN2+ Epi"
  ],
  "pyscenic": "0.12.1"
}
